# 검출: 공사 현장 PPE 11클래스

한 화면에 사람(Person)·안전모(helmet)·조끼(vest)가 여러 개 있다. 어디에 무엇이 있는지를 직사각형 박스로 잡으면, 몇 명이 모자를 썼는지 바로 셀 수 있다.

**런타임 → 런타임 유형 변경 → GPU(T4)** 를 먼저 고른다.

zip에는 학습·검증·테스트 그림과 학습해 둔 YOLO 가중치(`runs/train/weights/best.pt`)가 있다. 7번 평가와 8번 시연은 이 파일을 읽는다. 학습·추론 코드는 이 노트북 셀에 있다.
`/content`는 세션이 끝나면 비므로 zip은 드라이브에 남겨 둔다.


## 1. 데이터 다운로드

드라이브에서 해당 zip을 `링크가 있는 모든 사용자`로 공유한 뒤, 링크를 `ZIP_URL`에 넣는다.

파일이 100MB보다 크면 `wget`이 구글 확인 페이지를 받는다. 드라이브 링크는 `gdown`, 직접 열리는 주소는 `wget`을 쓴다.


In [ ]:
ZIP_URL = "https://drive.google.com/file/d/1rpq909ldsZ9paWCyOdBCXio_nYNFp49O/view?usp=sharing"

TASK = "object_detection"
ZIP_PATH = "/content/02_object_detection.zip"
DST = f"/content/{TASK}"


## 2. 패키지 다운로드


In [ ]:
!pip install -q gdown


In [ ]:
!pip install -q "ultralytics>=8.3,<9"
!apt-get -qq update
!apt-get -qq install -y fonts-noto-cjk fonts-nanum
from pathlib import Path
from PIL import ImageFont


def hangul_font(size=14):
    for path, index in (
        ("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc", 1),
        ("/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc", 1),
        ("/usr/share/fonts/truetype/nanum/NanumGothic.ttf", None),
        ("/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf", None),
    ):
        if not Path(path).exists():
            continue
        try:
            kw = {"index": index} if index is not None else {}
            font = ImageFont.truetype(path, size, **kw)
            print("hangul_font", path)
            return font
        except OSError:
            continue
    print("hangul_font missing. 범례는 영문으로 적는다.")
    return None


hangul_font()


## 3. zip 다운로드


In [ ]:
from pathlib import Path
import os


def fetch_zip(url, dest):
    url = (url or "").strip()
    if not url or "여기에" in url:
        raise SystemExit(
            "위 셀 ZIP_URL에 구글 드라이브 공유 링크(또는 직접 zip 주소)를 넣는다."
        )

    is_drive = "drive.google.com" in url or "docs.google.com" in url

    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 10_000:
        print("이미 받은 파일이 있다:", dest, dest.stat().st_size, "bytes")
        return

    dest.parent.mkdir(parents=True, exist_ok=True)

    os.environ["ZIP_URL"] = url
    os.environ["ZIP_PATH"] = str(dest)
    if is_drive:
        !gdown --fuzzy "$ZIP_URL" -O "$ZIP_PATH"
    else:
        !wget -c -O "$ZIP_PATH" "$ZIP_URL"

    print("받음", dest, dest.stat().st_size, "bytes")


fetch_zip(ZIP_URL, ZIP_PATH)


In [ ]:
!rm -rf {DST}


In [ ]:
!unzip -q -o {ZIP_PATH} -d /content


In [ ]:
from pathlib import Path
import torch
from IPython.display import display
from PIL import Image

ROOT = Path(DST)
if not ROOT.exists():
    raise SystemExit(f"{ROOT} 가 없다. zip 안에 과제 폴더가 있는지 확인한다.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ROOT", ROOT)
print("torch", torch.__version__)
print("device", DEVICE)
if DEVICE.type == "cuda":
    print("gpu", torch.cuda.get_device_name(0))


def show_img(img, width=900):
    """셀 결과에 PIL 이미지를 띄운다. 너무 가로로 긴 장만 줄인다."""
    img = img.convert("RGB")
    w, h = img.size
    if w > width:
        img = img.resize((width, int(h * width / w)), Image.BILINEAR)
    display(img)


## 4. Source Image와 Label 시각화

공사 현장에서 개인보호구(PPE, personal protective equipment)를 찾는 과제다. 한 화면에 사람(Person)과 안전모(helmet)·조끼(vest)·장갑(gloves)·안전화(boots)가 같이 있다. 
아래는 테스트 세트에서 고른 강의용 세 장이다. 왼쪽이 입력이고 오른쪽이 정답 박스다. 테스트 세트 전체는 `dataset/images/test`이고 평가는 7번에서 한다.
색 범례를 먼저 보고 세 장의 입력과 정답을 나란히 본다. 그 다음 숫자는 정답 라벨로 센 박스 개수다. 모델 점수가 아니라 한 장에 무엇이 몇 개 있는지를 보기 위한 숫자다.


In [ ]:
from collections import Counter
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

NUM_CLASSES = 11
CLASS_NAMES = [
    "helmet",
    "gloves",
    "vest",
    "boots",
    "goggles",
    "none",
    "Person",
    "no_helmet",
    "no_goggle",
    "no_gloves",
    "no_boots",
]
CLASS_KO = [
    "안전모",
    "장갑",
    "조끼",
    "안전화",
    "보안경",
    "기타",
    "사람",
    "안전모 없음",
    "보안경 없음",
    "장갑 없음",
    "안전화 없음",
]
CLASS_EN = [
    "helmet",
    "gloves",
    "vest",
    "boots",
    "goggles",
    "none",
    "Person",
    "no_helmet",
    "no_goggle",
    "no_gloves",
    "no_boots",
]
CLASS_LABEL = [
    ko if ko.lower() == en.lower() else f"{ko} ({en})"
    for ko, en in zip(CLASS_KO, CLASS_EN)
]
DET_COLORS = {
    0: (255, 210, 0),
    1: (0, 200, 180),
    2: (255, 130, 0),
    3: (160, 90, 30),
    4: (170, 90, 220),
    5: (140, 140, 140),
    6: (40, 110, 230),
    7: (220, 40, 40),
    8: (220, 40, 160),
    9: (255, 80, 120),
    10: (140, 20, 20),
}


def hangul_font(size):
    for path, index in (
        ("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc", 1),
        ("/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc", 1),
        ("/usr/share/fonts/truetype/nanum/NanumGothic.ttf", None),
        ("/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf", None),
    ):
        if not Path(path).exists():
            continue
        try:
            kw = {"index": index} if index is not None else {}
            return ImageFont.truetype(path, size, **kw)
        except OSError:
            continue
    return None


def font_and_names(size):
    found = hangul_font(size)
    if found is not None:
        return found, CLASS_LABEL
    for path in (
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
    ):
        try:
            return ImageFont.truetype(path, size), CLASS_EN
        except OSError:
            continue
    return ImageFont.load_default(), CLASS_EN


def font(size):
    found, _ = font_and_names(size)
    return found


def caption(img, text):
    bar_h = 28
    canvas = Image.new("RGB", (img.width, img.height + bar_h), (20, 20, 20))
    canvas.paste(img.convert("RGB"), (0, bar_h))
    draw = ImageDraw.Draw(canvas)
    draw.text((8, 6), text, fill=(255, 255, 255), font=font(16))
    return canvas


def side_by_side(left, right):
    w, h = left.size
    canvas = Image.new("RGB", (w * 2, h))
    canvas.paste(left, (0, 0))
    canvas.paste(right, (w, 0))
    return canvas


def legend_panel(height):
    f, names = font_and_names(14)
    title, _ = font_and_names(16)
    col_w = 300
    img = Image.new("RGB", (col_w, height), (20, 20, 20))
    draw = ImageDraw.Draw(img)
    draw.text((8, 8), "classes", fill=(255, 255, 255), font=title)
    gap = max(18, (height - 40) // NUM_CLASSES)
    for i, name in enumerate(names):
        y = 36 + i * gap
        draw.rectangle([8, y, 24, y + 14], fill=DET_COLORS[i])
        draw.text((32, y - 2), name, fill=(230, 230, 230), font=f)
    return img


def yolo_xyxy(line, width, height):
    cls, xc, yc, bw, bh = (float(p) for p in line.split())
    x1 = (xc - bw / 2.0) * width
    y1 = (yc - bh / 2.0) * height
    x2 = (xc + bw / 2.0) * width
    y2 = (yc + bh / 2.0) * height
    return int(cls), x1, y1, x2, y2


def gt_boxes(image):
    lab = image.parent / f"{image.stem}.txt"
    img = Image.open(image).convert("RGB")
    w, h = img.size
    boxes = []
    if lab.exists():
        for raw in lab.read_text().splitlines():
            if raw.strip():
                boxes.append(yolo_xyxy(raw, w, h))
    return boxes


print("클래스 색")
legend = ROOT / "examples" / "legend.png"
if legend.exists():
    show_img(Image.open(legend), width=360)
else:
    show_img(legend_panel(420), width=360)

print()
print("강의용 세 장. 왼쪽 입력, 오른쪽 정답")
for path in sorted((ROOT / "examples").glob("[0-9][0-9].jpg")):
    img = Image.open(path).convert("RGB")
    gt_path = path.parent / f"{path.stem}_gt.jpg"
    if gt_path.exists():
        gt_panel = Image.open(gt_path).convert("RGB")
        if gt_panel.size != img.size:
            gt_panel = gt_panel.resize(img.size, Image.BILINEAR)
    else:
        gt_panel = img.copy()
    show_img(
        side_by_side(caption(img, f"{path.name}  input"), caption(gt_panel, "GT")),
        width=980,
    )
    boxes = gt_boxes(path)
    counts = Counter(cls for cls, *_ in boxes)
    print(path.name, f"{img.size[0]}x{img.size[1]}  boxes {len(boxes)}")
    for c in range(NUM_CLASSES):
        n = int(counts.get(c, 0))
        if n:
            print(f"  {CLASS_LABEL[c]:24s}  {n}")
    print()


방금 본 장면은 object detection으로 푼다. 사람마다 몸통 박스 하나, 안전모·조끼·장갑·안전화마다 박스 하나를 그린다. 같은 사람이라도 장비는 따로 잡힌다. 
박스가 있으면 ‘이 장면에 안전모가 몇 개인가’, ‘조끼를 안 입은 사람이 있는가’를 개수로 바로 읽는다.
픽셀마다 칠하는 segmentation은 장비의 윤곽까지 남긴다. 이번 과제는 위치와 개수를 세는 일이라 직사각형이면 충분하다.
착용하지 않은 장비를 가리키는 클래스도 있다. `안전모 없음 (no_helmet)`처럼 이름에 없음이 들어가면, 그 장비를 사용하지 않은 위치를 박스로 표시한 것이다. 
강의용 세 장은 현장이 분명한 장만 골랐고, 테스트 세트 안에는 공사 현장이 아닌 장도 조금 섞여 있다.

## 5. 모델

입력 한 장을 넣으면 객체마다 클래스, 직사각형 좌표, 신뢰도가 나온다. 신뢰도가 0.25보다 낮은 후보는 버린다. 같은 물체를 가리키는 박스가 여러 개면 NMS가 겹침이 큰 쪽만 남긴다. 시연 그림은 그렇게 남은 박스를 그린 것이다.

YOLO는 그림을 여러 해상도의 격자로 나누고 칸마다 박스 후보를 낸다. 작은 안전모는 촘촘한 격자에서, 사람 몸통은 성긴 격자에서 잡히기 쉽다. Ultralytics YOLO26n은 COCO 80클래스로 사전학습된 nano 검출기다. 파라미터는 약 250만 개다. 이 과제에서는 헤드만 11클래스로 바꾸고 Construction-PPE 학습 장으로 이어서 학습한다.

`show_yolo_stack`이 그 경로를 그림으로 그린다.


In [ ]:
from io import BytesIO

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from PIL import Image
from ultralytics import YOLO


def show_yolo_stack():
    pretrained = ROOT / "weights" / "yolo26n.pt"
    model = YOLO(str(pretrained))
    n = sum(p.numel() for p in model.model.parameters())
    fig, ax = plt.subplots(figsize=(12.2, 4.6))
    ax.set_xlim(0, 12.4)
    ax.set_ylim(0.2, 4.4)
    ax.axis("off")
    ax.set_title("YOLO26n   image → boxes (class, xyxy, conf)", fontsize=14, pad=8)
    boxes = {}

    def box(key, cx, cy, title, sub, color, bw=2.05, bh=1.35):
        patch = FancyBboxPatch(
            (cx - bw / 2, cy - bh / 2),
            bw,
            bh,
            boxstyle="round,pad=0.02,rounding_size=0.1",
            facecolor=color,
            edgecolor="#222",
            linewidth=1.2,
        )
        ax.add_patch(patch)
        ax.text(cx, cy + 0.28, title, ha="center", va="center", fontsize=9, fontweight="bold", color="#111")
        ax.text(cx, cy - 0.28, sub, ha="center", va="center", fontsize=8, color="#222")
        boxes[key] = (cx, cy, bw, bh)

    def arrow(src, dst, text=""):
        x1, y1, w1, _ = boxes[src]
        x2, y2, w2, _ = boxes[dst]
        ax.annotate(
            "",
            xy=(x2 - w2 / 2 - 0.02, y2),
            xytext=(x1 + w1 / 2 + 0.02, y1),
            arrowprops=dict(arrowstyle="-|>", color="#333", lw=1.4, mutation_scale=10),
        )
        if text:
            ax.text((x1 + x2) / 2, y1 + 0.85, text, ha="center", fontsize=7.5, color="#444")

    box("in", 1.3, 2.4, "input", "640×640 RGB", "#ececec")
    box("back", 3.7, 2.4, "backbone", "stride 8 / 16 / 32", "#9ec0ea")
    box("neck", 6.1, 2.4, "neck", "fuse scales", "#6b5b95")
    box("head", 8.5, 2.4, "detect head", "11 classes here", "#2f9e66")
    box("out", 10.9, 2.4, "NMS boxes", "conf ≥ 0.25", "#ececec")
    arrow("in", "back")
    arrow("back", "neck")
    arrow("neck", "head")
    arrow("head", "out")
    ax.text(6.2, 0.55, f"params {n:,}   COCO pretrained nano, head remapped to PPE", ha="center", fontsize=9)
    fig.tight_layout()
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=140, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    buf.seek(0)
    show_img(Image.open(buf), width=980)


show_yolo_stack()


## 6. 학습

mAP50은 예측 박스와 정답 박스가 IoU 0.5 이상으로 겹칠 때 맞았다고 보고, 클래스마다 Average Precision을 구한 뒤 평균한 값이다. 0에서 1 사이다. 1에 가까울수록 위치와 종류가 정답과 잘 맞는다.
박스가 조금 빗나가도 같은 물체를 가리키면 IoU 0.5를 넘는다. 그래서 에폭마다 검증 세트의 mAP50을 보고 가장 높은 가중치를 남긴다. 검증은 가중치를 고르는 자리일 뿐이고 최종 점수는 아니다. 
테스트 장은 학습과 모델 선택에 쓰지 않는다.
이 셀은 공식 Construction-PPE 튜토리얼과 같이 5 epoch다. zip의 `runs/train/weights/best.pt`는 덮지 않는다. 7번 평가와 8번 시연은 그 파일을 읽는다. 시연만 하면 이 셀을 건너뛴다.


In [ ]:
from pathlib import Path
from ultralytics import YOLO

EPOCHS = 5
IMGSZ = 640

pretrained = ROOT / "weights" / "yolo26n.pt"
if not pretrained.exists():
    raise SystemExit(f"사전학습 가중치가 없다: {pretrained}")

data_yaml = ROOT / "runs" / "data_resolved.yaml"
data_yaml.parent.mkdir(parents=True, exist_ok=True)
names = "\n".join(f"  {i}: {n}" for i, n in enumerate(CLASS_NAMES))
data_yaml.write_text(
    f"path: {ROOT / 'dataset'}\n"
    "train: images/train\n"
    "val: images/val\n"
    "test: images/test\n"
    f"names:\n{names}\n",
    encoding="utf-8",
)

out_dir = ROOT / "runs" / "train_colab"
model = YOLO(str(pretrained))
model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    project=str(ROOT / "runs"),
    name="train_colab",
    exist_ok=True,
)
print("saved", out_dir / "weights" / "best.pt")
print("zip best.pt unchanged", ROOT / "runs" / "train" / "weights" / "best.pt")


## 7. 테스트 평가

학습과 체크포인트에 쓰지 않은 테스트 데이터셋을 사용6번에서 만든 가중치를 보려면 `WEIGHTS`를 `runs/train_colab/weights/best.pt`로 바꾼다.
첫 줄의 `mAP50`은 클래스마다 AP50을 구한 뒤 평균한 값이다. `mAP50-95`는 IoU 문턱을 0.5부터 0.95까지 올려 가며 더 빡세게 본 값이라 더 낮다.
그 아래 줄은 클래스마다 AP50이다. 높으면 그 사람이나 장비를 맞게 잡은 것이고, 낮으면 빠지거나 다른 종류로 넘친 것이다. 평균은 클래스마다 한 번씩 넣기 때문에 사람처럼 자주 나오는 클래스와 보안경처럼 드문 클래스가 같은 비중으로 들어간다. 
테스트 안에 공사 현장이 아닌 장이 있으면 `안전모 없음 (no_helmet)` 점수가 내려가고 평균도 같이 내려간다.

In [ ]:
import json
from pathlib import Path
from ultralytics import YOLO

WEIGHTS = ROOT / "runs" / "train" / "weights" / "best.pt"
print("weights", WEIGHTS)
data_yaml = ROOT / "runs" / "data_resolved.yaml"
if not data_yaml.exists():
    data_yaml.parent.mkdir(parents=True, exist_ok=True)
    names = "\n".join(f"  {i}: {n}" for i, n in enumerate(CLASS_NAMES))
    data_yaml.write_text(
        f"path: {ROOT / 'dataset'}\n"
        "train: images/train\n"
        "val: images/val\n"
        "test: images/test\n"
        f"names:\n{names}\n",
        encoding="utf-8",
    )

model = YOLO(str(WEIGHTS))
metrics = model.val(
    data=str(data_yaml),
    split="test",
    imgsz=640,
    plots=False,
)
test_dir = ROOT / "dataset" / "images" / "test"
n_img = sum(1 for p in test_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
maps = list(metrics.box.ap50)
print(
    f"held_out_test  n={n_img}  "
    f"mAP50 {float(metrics.box.map50):.3f}  mAP50-95 {float(metrics.box.map):.3f}"
)
per = {}
for i, name in enumerate(CLASS_NAMES):
    ap = float(maps[i]) if i < len(maps) else 0.0
    label = CLASS_LABEL[i]
    per[label] = round(ap, 4)
    print(f"  {label:24s}  AP50 {ap:.3f}")
report = {
    "n": n_img,
    "n_classes": int(metrics.box.nc),
    "map50": round(float(metrics.box.map50), 4),
    "map50_95": round(float(metrics.box.map), 4),
    "precision": round(float(metrics.box.mp), 4),
    "recall": round(float(metrics.box.mr), 4),
    "per_class_ap50": per,
}
out = ROOT / "runs" / "test_metrics.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("saved", out)


## 8. 추론

왼쪽부터 입력, 정답, YOLO 예측, 7번과 같이 zip의 `best.pt`를 읽는다. 6번에서 만든 가중치를 보려면 `WEIGHTS`를 `runs/train_colab/weights/best.pt`로 바꾼다. 사람(Person)·안전모(helmet)·조끼(vest)는 테스트셋에서 잘 맞춘다. 여기 숫자는 이 장에서 센 박스 개수이고, 테스트 세트 점수와 바꿔 읽지 않는다.

In [ ]:
from collections import Counter
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO


def draw_boxes(img, boxes, with_conf):
    canvas = img.convert("RGB").copy()
    draw = ImageDraw.Draw(canvas)
    f = font(14)
    counts = {name: 0 for name in CLASS_NAMES}
    for item in boxes:
        cls, x1, y1, x2, y2 = item[:5]
        conf = item[5] if len(item) > 5 else None
        if not (0 <= cls < NUM_CLASSES):
            continue
        color = DET_COLORS[cls]
        name = CLASS_NAMES[cls]
        counts[name] += 1
        label = CLASS_EN[cls] if (conf is None or not with_conf) else f"{CLASS_EN[cls]} {conf:.2f}"
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        tw = max(8 * len(label), 40)
        draw.rectangle([x1, max(0, y1 - 18), x1 + tw, y1], fill=color)
        draw.text((x1 + 2, max(0, y1 - 17)), label, fill=(0, 0, 0), font=f)
    return canvas, counts


def pred_boxes(result):
    out = []
    if result.boxes is None:
        return out
    for b in result.boxes:
        cls = int(b.cls.item())
        conf = float(b.conf.item())
        x1, y1, x2, y2 = (float(v) for v in b.xyxy[0].tolist())
        out.append((cls, x1, y1, x2, y2, conf))
    return out


WEIGHTS = ROOT / "runs" / "train" / "weights" / "best.pt"
print("weights", WEIGHTS)
model = YOLO(str(WEIGHTS))
images = sorted((ROOT / "examples").glob("[0-9][0-9].jpg"))
results = model.predict(
    source=[str(p) for p in images],
    conf=0.25,
    verbose=False,
    save=False,
)
out_dir = ROOT / "runs" / "infer"
out_dir.mkdir(parents=True, exist_ok=True)

for path, r in zip(images, results):
    raw = Image.open(path).convert("RGB")
    if r.orig_shape is not None:
        oh, ow = int(r.orig_shape[0]), int(r.orig_shape[1])
        view = raw.resize((ow, oh), Image.BILINEAR) if raw.size != (ow, oh) else raw
    else:
        view = raw
    gt = draw_boxes(view, gt_boxes(path), with_conf=False)[0]
    pred = pred_boxes(r)
    pred_img, counts = draw_boxes(view, pred, with_conf=True)
    panels = [
        caption(view, "input"),
        caption(gt, "GT"),
        caption(pred_img, "YOLO"),
    ]
    w, h = panels[0].size
    leg = legend_panel(h)
    canvas = Image.new("RGB", (w * 3 + leg.width, h))
    for i, panel in enumerate(panels):
        canvas.paste(panel, (i * w, 0))
    canvas.paste(leg, (w * 3, 0))
    out = out_dir / f"{path.stem}_pred.jpg"
    canvas.save(out, quality=92)
    gt_counts = Counter(cls for cls, *_ in gt_boxes(path))
    print(path.name)
    print("  GT")
    for c in range(NUM_CLASSES):
        n = int(gt_counts.get(c, 0))
        if n:
            print(f"    {CLASS_LABEL[c]:24s}  {n}")
    print("  pred")
    for name in CLASS_NAMES:
        if counts[name]:
            print(f"    {CLASS_LABEL[CLASS_NAMES.index(name)]:24s}  {counts[name]}")
    show_img(canvas, width=980)
    print()
